In [ ]:
# 🛠️ PIPELINE DEPENDENCIES (LOCKED & STABLE ECOSYSTEM)

# 1. Kunci Numpy di versi stabil
!pip install numpy==1.26.4

# 2. PyTorch CUDA & Akselerator P104-100
!pip install torch==2.3.0 torchvision==0.18.0 accelerate==0.30.1

# 3. Transformers & Text Processing
!pip install transformers==4.41.2 tiktoken==0.7.0 einops==0.8.0

# 4. Render PDF & Image
!pip install pdf2image==1.17.0 pillow==10.3.0 verovio==3.15.0 pypdf

# 5. Ekosistem LangChain 0.2.x (Kunci langchain-core < 0.3.0)
!pip install "langchain-core<0.3.0" langchain==0.2.5 langchain-community==0.2.5 langchain-qdrant==0.1.1 langchain-ollama==0.1.0 qdrant-client==1.9.0

In [ ]:
import socket

# 🛠️ PIPELINE DEPENDENCIES (LOCKED & STABLE ECOSYSTEM)

# 1. Kunci Numpy di versi stabil
!pip install numpy==1.26.4

# 2. PyTorch CUDA & Akselerator P104-100
!pip install torch==2.3.0 torchvision==0.18.0 accelerate==0.30.1

# 3. Transformers & Text Processing
!pip install transformers==4.41.2 tiktoken==0.7.0 einops==0.8.0

# 4. Render PDF & Image
!pip install pdf2image==1.17.0 pillow==10.3.0 verovio==3.15.0 pypdf

# 5. Ekosistem LangChain 0.2.x (Kunci langchain-core < 0.3.0)
!pip install "langchain-core<0.3.0" langchain==0.2.5 langchain-community==0.2.5 langchain-qdrant==0.1.1 langchain-ollama==0.1.0 qdrant-client==1.9.0

✅ Port 11434 di host.docker.internal TERBUKA BANGET!
👉 Masalahnya MURNI di library 'requests' / Proxy Environment Python.


In [ ]:
import os
import glob
import json
import gc
import tempfile
import requests
import torch
from pdf2image import convert_from_path, pdfinfo_from_path
from transformers import AutoModel, AutoTokenizer
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings

try:
    from langchain_qdrant import QdrantVectorStore
except ImportError:
    from langchain_qdrant import Qdrant as QdrantVectorStore

from qdrant_client import QdrantClient

# Paksa Python untuk TIDAK lewat proxy kalau nembak IP Tailscale & Localhost
os.environ["NO_PROXY"] = "100.80.128.112,localhost,127.0.0.1,172.*"
os.environ["no_proxy"] = "100.80.128.112,localhost,127.0.0.1,172.*"

# (Opsional) Hapus variabel proxy jika ada
os.environ.pop("HTTP_PROXY", None)
os.environ.pop("HTTPS_PROXY", None)
os.environ.pop("http_proxy", None)
os.environ.pop("https_proxy", None)

# --- CONFIGURASI IP ---
# JANGAN PAKAI 127.0.0.1 kalau dari WSL2/Container ke Windows Host!
# Gunakan IP vEthernet WSL / IP Local PC / IP Tailscale
IP_WINDOWS_LU = "host.docker.internal"  # <-- GANTI SESUAI IP HOST WINDOWS / GATEWAY
FOLDER_DATA = "./data_awal"
TONG_SAMPAH_F = "/tmp/tong_sampah_emi"
FILE_RESUME = "resume_log.json"
COLLECTION_NAME = "emi_knowledge"

# --- 1. HEALTH CHECK OLLAMA & QDRANT SEBELUM OCR 2 JAM ---
def ping_service(url, name):
    try:
        # Tambahkan proxies={"http": None, "https": None}
        r = requests.get(url, timeout=5, proxies={"http": None, "https": None})
        print(f"✅ {name} Online! status: {r.status_code}")
    except Exception as e:
        raise ConnectionError(f"❌ CRITICAL: {name} TIDAK BISA DIAKSES di {url}.\nDetail: {e}")

print("🔍 Checking services connection...")
ping_service(f"http://{IP_WINDOWS_LU}:11434/api/tags", "Ollama")
ping_service(f"http://{IP_WINDOWS_LU}:6333/readyz", "Qdrant")

# --- INIT MODELS ---
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
tokenizer = AutoTokenizer.from_pretrained('stepfun-ai/GOT-OCR2_0', trust_remote_code=True)
model = AutoModel.from_pretrained(
    'stepfun-ai/GOT-OCR2_0',
    trust_remote_code=True,
    torch_dtype=torch.float16
).to('cuda').eval()

embeddings = OllamaEmbeddings(
    model="bge-m3:latest",
    base_url=f"http://{IP_WINDOWS_LU}:11434"
)
qdrant_client = QdrantClient(url=f"http://{IP_WINDOWS_LU}:6333", timeout=120)

try:
    vector_store = QdrantVectorStore(client=qdrant_client, collection_name=COLLECTION_NAME, embedding=embeddings)
except TypeError:
    vector_store = QdrantVectorStore(client=qdrant_client, collection_name=COLLECTION_NAME, embeddings=embeddings)

headers_to_split_on = [("#", "Header 1"), ("##", "Header 2"), ("###", "Header 3")]
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)

def load_resume_log():
    if os.path.exists(FILE_RESUME):
        with open(FILE_RESUME, 'r') as f:
            return json.load(f)
    return []

def save_resume_log(pdf_name, log_data):
    if pdf_name not in log_data:
        log_data.append(pdf_name)
        with open(FILE_RESUME, 'w') as f:
            json.dump(log_data, f)
        print(f"   💾 [CHECKPOINT] {pdf_name} tersimpan di log!")

# --- PROSES OCR DENGAN INJEKSI LANGSUNG PER-BATCH ---
def proses_dan_injeksi_scan(file_path):
    info = pdfinfo_from_path(file_path)
    total_pages = info["Pages"]
    BATCH_SIZE = 15

    print(f"   📄 Total Halaman Scan: {total_pages}. Memproses via GOT-OCR (GPU Locked)...")

    for start_page in range(1, total_pages + 1, BATCH_SIZE):
        end_page = min(start_page + BATCH_SIZE - 1, total_pages)
        print(f"   📸 [BATCH RENDER] Membaca Halaman {start_page}-{end_page}/{total_pages}...")

        images = convert_from_path(
            file_path, dpi=150, first_page=start_page, last_page=end_page, fmt="jpeg"
        )

        batch_docs = []
        for idx, img in enumerate(images):
            current_page = start_page + idx
            with tempfile.NamedTemporaryFile(suffix=".jpg", dir=TONG_SAMPAH_F, delete=True) as tmp_img:
                img.save(tmp_img.name, "JPEG")
                try:
                    with torch.no_grad():
                        res = model.chat(tokenizer, tmp_img.name, ocr_type='format')
                    if res and res.strip():
                        batch_docs.append(
                            Document(
                                page_content=res.strip(),
                                metadata={"source": os.path.basename(file_path), "page_number": current_page}
                            )
                        )
                        print(f"   ⚡ [GPU HIT] Hal {current_page}/{total_pages} Selesai!")
                except Exception as e:
                    print(f"   ❌ [GOT-OCR Error Hal {current_page}]: {e}")

        # INJEKSI LANGSUNG PER-BATCH (Mencegah Kehilangan Data Jika Crash di Tengah Jalan)
        if batch_docs:
            chunks_to_inject = []
            for doc in batch_docs:
                md_splits = markdown_splitter.split_text(doc.page_content)
                for md_doc in md_splits:
                    md_doc.metadata.update(doc.metadata)
                sub_chunks = recursive_splitter.split_documents(md_splits if md_splits else [doc])
                chunks_to_inject.extend(sub_chunks)

            print(f"   💉 [STREAM INJECT] Menyuntikkan {len(chunks_to_inject)} chunk (Hal {start_page}-{end_page}) ke Qdrant...")
            vector_store.add_documents(documents=chunks_to_inject)

        torch.cuda.empty_cache()
        gc.collect()

def main():
    print("🚀 MEMULAI INJEKSI MEMORI EMI...")
    if not os.path.exists(TONG_SAMPAH_F):
        os.makedirs(TONG_SAMPAH_F)

    file_list = glob.glob(os.path.join(FOLDER_DATA, "*.pdf"))
    resume_log = load_resume_log()

    for file_path in file_list:
        nama_file = os.path.basename(file_path)
        if nama_file in resume_log:
            print(f"\n⏩ [SKIP] {nama_file} sudah diinjeksi sebelumnya.")
            continue

        print(f"\n📄 Memproses File: {nama_file}")
        loader = PyPDFLoader(file_path)
        halaman_pdf = loader.load()
        teks_uji = halaman_pdf[0].page_content.strip() if halaman_pdf else ""

        if len(teks_uji) < 50:
            print("   ⚠️ [SCAN / TABEL DETECTED] Menyerahkan ke GOT-OCR 2.0 GPU!")
            proses_dan_injeksi_scan(file_path)
            save_resume_log(nama_file, resume_log)
        else:
            print("   ✅ PDF Digital terdeteksi. Sedot teks langsung!")
            final_chunks = recursive_splitter.split_documents(halaman_pdf)
            if final_chunks:
                vector_store.add_documents(documents=final_chunks)
                save_resume_log(nama_file, resume_log)

    print("\n✅ PIPELINE SELESAI!")

if __name__ == "__main__":
    main()

🔍 Checking services connection...
✅ Ollama Online! status: 200
✅ Qdrant Online! status: 200
🚀 MEMULAI INJEKSI MEMORI EMI...

📄 Memproses File: CakBun Indo 2024.pdf
   ⚠️ [SCAN / TABEL DETECTED] Menyerahkan ke GOT-OCR 2.0 GPU!
   📄 Total Halaman Scan: 150. Memproses via GOT-OCR (GPU Locked)...
   📸 [BATCH RENDER] Membaca Halaman 1-15/150...


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


   ⚡ [GPU HIT] Hal 1/150 Selesai!


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


   ⚡ [GPU HIT] Hal 2/150 Selesai!


...

   ⚡ [GPU HIT] Hal 112/115 Selesai!


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


   ⚡ [GPU HIT] Hal 113/115 Selesai!


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


   ⚡ [GPU HIT] Hal 114/115 Selesai!
   ⚡ [GPU HIT] Hal 115/115 Selesai!
   💉 [STREAM INJECT] Menyuntikkan 27 chunk (Hal 106-115) ke Qdrant...
   💾 [CHECKPOINT] Fix Buku CakBun 2026.pdf tersimpan di log!

✅ PIPELINE SELESAI!
